# 05 — A demanding case: Census income, end to end

*Chapter 09 · XGBoost · Notebook 5 of 5*

This is the chapter's capstone: a full, honest workflow on a real, messy tabular dataset that pulls
together everything you built. You will look before modelling, meet genuine missing values and put
Notebook 3's native handling to an honest test, compare XGBoost against the whole field on equal terms,
tune it with early stopping, read its precision and recall at a chosen threshold, ask what really drives
it, and end by confronting what the model should — and should not — be used for.

The dataset is **Adult / Census Income**: predict whether a 1994 US census respondent earned more than
$50,000, from demographic and work features. It is a classic teaching set precisely because it is
awkward in instructive ways — mixed numeric and categorical columns, real missing values, class
imbalance, and a social-bias dimension we will not look away from.

**Prerequisites.** All of Chapter 09 (NB 1–4); Chapter 00 (train/test, the stratified split,
cross-validation, accuracy vs precision/recall, ROC/PR, the threshold); Chapters 06/08 (the
gain-importance caveat, permutation importance).

**What you'll be able to do.**
- Run an honest, leakage-free workflow on imbalanced tabular data and report PR-AUC, not accuracy alone.
- Test whether native missing-value handling actually helps — and read a null result honestly.
- Compare models on equal terms, naming the preprocessing each one needs.
- Tune with early stopping, choose a decision threshold, and read importances with permutation.
- State a model's limits and ethical boundaries plainly.

## The problem, and why it is a good capstone

We predict a binary label — income `>50K` vs `<=50K` — from 14 features: some numeric (age, hours per
week, capital gains), some categorical (occupation, education, marital status). Four things make this a
real test rather than a toy:

- **Mixed types** — categoricals XGBoost can take natively (NB 4) but most models cannot.
- **Genuine missing values** — three columns have gaps, the case Notebook 3 was built for.
- **Class imbalance** — only about a quarter of people earn `>50K`, so accuracy will mislead us.
- **A real ethics dimension** — the target is income, predicted from demographics, in 1994 US data.

That last point is not decoration. We will measure the model's behaviour across demographic groups and
be explicit, at the end, about why this model is a teaching example and **not** something to deploy.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.ensemble import (
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

from ml_course import viz
from ml_course.colors import CMAP_COUNT, COLORS

viz.use_course_style()
SEED = 0

adult = fetch_openml("adult", version=2, as_frame=True)
X = adult.data.copy()
y = (adult.target == ">50K").astype(int)   # 1 = earns >50K
num_cols = ["age", "fnlwgt", "education-num", "capital-gain", "capital-loss", "hours-per-week"]
cat_cols = ["workclass", "education", "marital-status", "occupation", "relationship",
            "race", "sex", "native-country"]
miss_cols = ["workclass", "occupation", "native-country"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)
print(f"shape {X.shape}   positive rate P(>50K) = {y.mean():.4f}")
print(f"{len(num_cols)} numeric + {len(cat_cols)} categorical columns")
print("missing (NaN): " + ", ".join(f"{c} {int(X[c].isna().sum())}" for c in miss_cols))
print(f"stratified split: train {X_train.shape[0]} / test {X_test.shape[0]}")


## Look before you model — the class balance

The first question on any classification task: how balanced are the labels? It decides which metric we
can trust.

In [ ]:
counts = y.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.bar(["<=50K", ">50K"], counts.values, color=[COLORS["class_a"], COLORS["class_b"]])
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{v} ({v / len(y):.1%})", ha="center", va="bottom", color=COLORS["text"])
ax.set_ylabel("count")
ax.set_ylim(0, counts.max() * 1.15)
ax.set_title("class balance — Adult / Census income")
plt.show()
print(counts.to_string())


**Read the figure.** About **24%** of people earn `>50K`; the other 76% do not. That imbalance is
the reason accuracy will mislead us: a model that predicts `<=50K` for *everyone* already scores 0.76,
while being useless at finding high earners. So throughout this notebook we lead with **PR-AUC** (the
area under the precision–recall curve, also called average precision), which focuses on the minority
class we actually care about, and we watch precision and recall at a chosen threshold (Chapter 00).

## Missing values — where, and are they informative?

Three columns carry real gaps: `workclass`, `occupation`, and `native-country`. Before deciding how to
handle them, ask the question Notebook 3 raised: is the *fact* of being missing itself informative? We
compare the rate of `>50K` among rows where a value is missing against rows where it is present.

In [ ]:
base = y.mean()
p_missing = [y[X[c].isna()].mean() for c in miss_cols]
p_present = [y[~X[c].isna()].mean() for c in miss_cols]
xpos = np.arange(len(miss_cols))
w = 0.38
fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.bar(xpos - w / 2, p_missing, w, color=COLORS["error"], label="value missing")
ax.bar(xpos + w / 2, p_present, w, color=COLORS["model"], label="value present")
ax.axhline(base, color=COLORS["muted"], linestyle="--", label=f"base rate {base:.3f}")
ax.set_xticks(xpos)
ax.set_xticklabels(miss_cols)
ax.set_ylabel("P(income > 50K)")
ax.set_title("is missingness informative?")
ax.legend()
plt.show()
for c, pm, pp in zip(miss_cols, p_missing, p_present, strict=True):
    print(f"  {c:16s}: missing {pm:.4f}   present {pp:.4f}")


**Read the figure.** For `workclass` and `occupation`, a missing value is strongly informative:
those rows earn `>50K` only about **9%** of the time, versus **25%** when the value is present — these
are largely people outside the paid labour force. (`native-country` missing sits marginally *above* the
base rate — a weak, opposite-direction effect we set aside.) So the missingness *looks* genuinely useful
— which makes Adult the perfect place to ask, head on, whether XGBoost's native handling (Notebook 3)
actually earns its keep here.

## The honest test — does native handling beat imputation?

Notebook 3 showed XGBoost routes missing rows down a learned default direction, with no imputation. The
natural hypothesis: since missingness is informative here, keeping the `NaN` (native) should beat
filling it in (imputation, which erases the missing flag). Let us **measure** it rather than assume.
Same XGBoost three times, changing only how the three columns' gaps are treated:

- **native** — leave them `NaN` (Notebook 3's handling);
- **mode-imputed** — fill each gap with the column's most frequent value (erasing the missing flag);
- **explicit "missing"** — add a literal `MISSING` category (which *also* preserves the signal).

In [ ]:
xgb_kw = dict(enable_categorical=True, tree_method="hist", n_estimators=300,
                  max_depth=6, learning_rate=0.1, random_state=SEED)


def pr_auc(model, Xte):
    return average_precision_score(y_test, model.predict_proba(Xte)[:, 1])


# native: NaN kept
m_native = XGBClassifier(**xgb_kw).fit(X_train, y_train)
ap_native = pr_auc(m_native, X_test)

# mode-imputed: fill missing categoricals with the TRAIN mode (erase the flag)
X_tr_imp, X_te_imp = X_train.copy(), X_test.copy()
for c in miss_cols:
    mode = X_train[c].mode(dropna=True)[0]
    X_tr_imp[c] = X_tr_imp[c].fillna(mode)
    X_te_imp[c] = X_te_imp[c].fillna(mode)
ap_imp = pr_auc(XGBClassifier(**xgb_kw).fit(X_tr_imp, y_train), X_te_imp)

# explicit 'missing' category (also preserves the signal)
X_tr_f, X_te_f = X_train.copy(), X_test.copy()
for c in miss_cols:
    X_tr_f[c] = X_tr_f[c].cat.add_categories(["MISSING"]).fillna("MISSING")
    X_te_f[c] = X_te_f[c].cat.add_categories(["MISSING"]).fillna("MISSING")
ap_flag = pr_auc(XGBClassifier(**xgb_kw).fit(X_tr_f, y_train), X_te_f)

aps = [ap_native, ap_imp, ap_flag]
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(["native (NaN)", "mode-imputed", "explicit 'missing'"], aps,
       color=[COLORS["model"], COLORS["muted"], COLORS["class_c"]])
for i, v in enumerate(aps):
    ax.text(i, v, f"{v:.4f}", ha="center", va="bottom", color=COLORS["text"])
ax.set_ylim(0, 1.0)
ax.set_ylabel("test PR-AUC (average precision)")
ax.set_title("does native missing-handling move the needle?")
plt.show()
print(f"native {ap_native:.4f}   mode-imputed {ap_imp:.4f}   explicit {ap_flag:.4f}")
print(f"delta (native - imputed) = {ap_native - ap_imp:+.4f}")

# Is that -0.001 a real deficit or split-noise? Two diagnostics.
# (a) the two informative-missing columns are nearly the SAME flag:
both = int((X["occupation"].isna() & X["workclass"].isna()).sum())
n_occ = int(X["occupation"].isna().sum())
print(f"occupation-missing rows also workclass-missing: {both} of {n_occ}")
# (b) re-measure the native-imputed gap across 5 resplits -- does it even keep its sign?
deltas = []
for s in range(5):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.20, stratify=y, random_state=s)
    nm = XGBClassifier(**xgb_kw).fit(Xa, ya)
    nat = average_precision_score(yb, nm.predict_proba(Xb)[:, 1])
    Xa_i, Xb_i = Xa.copy(), Xb.copy()
    for c in miss_cols:
        mo = Xa[c].mode(dropna=True)[0]
        Xa_i[c] = Xa_i[c].fillna(mo)
        Xb_i[c] = Xb_i[c].fillna(mo)
    im = XGBClassifier(**xgb_kw).fit(Xa_i, ya)
    imp = average_precision_score(yb, im.predict_proba(Xb_i)[:, 1])
    deltas.append(nat - imp)
deltas = np.array(deltas)
print(f"native - imputed across 5 resplits: mean {deltas.mean():+.4f}  sd {deltas.std():.4f}  "
      f"range [{deltas.min():+.4f}, {deltas.max():+.4f}]")


**Read the figure.** Here is the honest surprise: the three bars are **the same to within noise**
(PR-AUC ≈ 0.83). The resampling check makes that precise — across five train/test splits the
native-minus-imputed gap averages essentially zero and even **flips sign** (here it is −0.001; on other
splits it is positive), so the deficit on this one split is split-to-split noise, not a real effect.
Native handling does **not** beat imputation on this dataset.

Why, when missingness looked so informative? The data shows two reasons. First, the two
informative-missing columns are **nearly the same flag**: almost every row missing `occupation` is also
missing `workclass` (2,799 of 2,809), so they are not two independent signals. Second, what that flag
carries is **largely** also present in the rest of the features — `relationship`, `marital-status`, low
`capital-gain` — so the small unique signal it adds does not register in PR-AUC at this resolution. The
lesson is the one this whole course keeps teaching: **measure the lever, do not assume it.**
Informative-*looking* missingness does not guarantee native handling wins.

So what *is* native handling worth here? **Convenience.** XGBoost (and HistGBR) take the `NaN` and the
categoricals with no preprocessing at all — and the next comparison shows exactly what the models that
*cannot* do that are forced to add.

## The cross-method comparison — on equal, disclosed terms

How does XGBoost compare with the rest of the field? A comparison is only fair if we state the
preprocessing each model needs — because part of any difference is the preprocessing, not the model:

- **XGBoost, HistGBR** — take categoricals and `NaN` **natively**; no preprocessing.
- **GradientBoosting, RandomForest, DecisionTree** — cannot take `NaN` or strings, so we **ordinal-encode
  + impute** the categoricals.
- **Logistic regression** — needs **one-hot encoding + scaling** (and imputation; rare categories pooled) for its linear form.

All six see the same split. We also fit **LightGBM** (Chapter 10) as a teaser.

In [ ]:
ord_imp = ColumnTransformer([
    ("num", "passthrough", num_cols),
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1,
                           encoded_missing_value=-2), cat_cols),
])
ohe_scale = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=10), cat_cols),
])


def fit_ap_acc(model, Xtr, Xte):
    model.fit(Xtr, y_train)
    return pr_auc(model, Xte), accuracy_score(y_test, model.predict(Xte))


results = {}
results["XGBoost (native)"] = (ap_native, accuracy_score(y_test, m_native.predict(X_test)))
results["HistGBR (native)"] = fit_ap_acc(
    HistGradientBoostingClassifier(categorical_features="from_dtype", random_state=SEED,
                                   max_iter=300, learning_rate=0.1), X_train, X_test)
results["GradientBoosting (imputed)"] = fit_ap_acc(
    Pipeline([("prep", ord_imp), ("clf", GradientBoostingClassifier(random_state=SEED))]),
    X_train, X_test)
results["RandomForest (imputed)"] = fit_ap_acc(
    Pipeline([("prep", ord_imp),
              ("clf", RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1))]),
    X_train, X_test)
results["Logistic (one-hot+scale)"] = fit_ap_acc(
    Pipeline([("prep", ohe_scale), ("clf", LogisticRegression(max_iter=1000, random_state=SEED))]),
    X_train, X_test)
results["DecisionTree d4 (imputed)"] = fit_ap_acc(
    Pipeline([("prep", ord_imp), ("clf", DecisionTreeClassifier(max_depth=4, random_state=SEED))]),
    X_train, X_test)

# LightGBM teaser (Chapter 10) — native categoricals + NaN; its training log is left visible.
t0 = time.perf_counter()
lgbm = LGBMClassifier(n_estimators=300, learning_rate=0.1, random_state=SEED).fit(X_train, y_train)
ap_lgbm, dt_lgbm = pr_auc(lgbm, X_test), time.perf_counter() - t0

ranked = sorted(results.items(), key=lambda kv: -kv[1][0])
native = {"XGBoost (native)", "HistGBR (native)"}
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.barh([n for n, _ in ranked], [v[0] for _, v in ranked],
        color=[COLORS["model"] if n in native else COLORS["muted"] for n, _ in ranked])
for i, (_, v) in enumerate(ranked):
    ax.text(v[0], i, f" {v[0]:.3f}", va="center", color=COLORS["text"])
ax.set_xlim(0, 1.0)
ax.invert_yaxis()
ax.set_xlabel("test PR-AUC")
ax.set_title("cross-method (blue = native NaN+categoricals, grey = must impute+encode)")
plt.show()
print(f"  {'model':30s} {'PR-AUC':>8s} {'acc':>8s}")
for n, (ap, acc) in ranked:
    print(f"  {n:30s} {ap:8.4f} {acc:8.4f}")
print(f"  LightGBM teaser (native): PR-AUC {ap_lgbm:.4f}  fit {dt_lgbm:.2f}s")


**Read the figure.** The two native boosters lead — **HistGBR ≈ 0.829, XGBoost ≈ 0.828** — with
plain GradientBoosting a step behind (≈ 0.814), then RandomForest (≈ 0.788), Logistic regression (≈ 0.768),
and the shallow tree far back (≈ 0.668). LightGBM lands right with the leaders (≈ 0.828, in about two
seconds).

Two honest readings. First, the boosters win, but the margin over GradientBoosting is small — and part
of even that gap is **convenience**: XGBoost and HistGBR skipped the ordinal-encode-and-impute step the
others needed, which is a real practical edge but not raw modelling power. Second, **there is no
universal best**: on this data XGBoost's advantage is speed, native handling of missing values and
categoricals, and the regularization you learned to control in Notebook 4 — *not* a dramatic jump in
PR-AUC. That is the honest headline of the whole chapter, now measured on a real problem.

## Tuning honestly — early stopping, and the test opened once

To tune, we follow Chapter 00's discipline: carve a **validation** slice out of the training data, let
the model use it to decide *how many trees* to grow (early stopping), and open the **test set only once**
at the very end. XGBoost's 3.x API puts `early_stopping_rounds` and `eval_metric` in the **constructor**,
and the `eval_set` in `.fit()`. We request a generous 2000 trees and let the data stop us.

In [ ]:
X_tr2, X_val, y_tr2, y_val = train_test_split(
    X_train, y_train, test_size=0.20, stratify=y_train, random_state=SEED)
tuned = XGBClassifier(
    enable_categorical=True, tree_method="hist", n_estimators=2000, learning_rate=0.05,
    max_depth=6, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    early_stopping_rounds=50, eval_metric="aucpr", random_state=SEED)
# verbose=50 prints the validation score every 50 rounds (progress kept visible, not silenced).
tuned.fit(X_tr2, y_tr2, eval_set=[(X_val, y_val)], verbose=50)

evals = tuned.evals_result()["validation_0"]["aucpr"]
best = tuned.best_iteration
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(range(1, len(evals) + 1), evals, color=COLORS["model"], linewidth=2)
ax.axvline(best + 1, color=COLORS["highlight"], linestyle="--", label=f"best_iteration = {best}")
ax.set_xlabel("number of trees")
ax.set_ylabel("validation PR-AUC")
ax.set_title("early stopping (2000 trees requested)")
ax.legend()
plt.show()
ap_tuned = pr_auc(tuned, X_test)
acc_tuned = accuracy_score(y_test, tuned.predict(X_test))
roc_tuned = roc_auc_score(y_test, tuned.predict_proba(X_test)[:, 1])
print(f"best_iteration = {best} (of 2000 requested)")
print(f"tuned test: PR-AUC {ap_tuned:.4f}   acc {acc_tuned:.4f}   ROC-AUC {roc_tuned:.4f}")


**Read the figure.** The validation PR-AUC climbs fast, then flattens; early stopping halts the
2000-tree request at **best_iteration ≈ 263** — the point past which 50 more rounds brought no
improvement. The tuned model scores test **PR-AUC ≈ 0.829**, in line with the leaders above. You did not
guess the number of trees; you set a generous budget and let a held-out slice choose it (Chapter 08's
lesson, in XGBoost's constructor-parameter form).

## Reading the classifier — precision, recall, and the threshold

A probability model does not decide anything until you pick a **threshold**. The default 0.5 is only one
choice, and on imbalanced data rarely the right one. We read the full precision–recall curve and two
operating points: the default 0.5, and a threshold that maximises F1 (a balance of precision and
recall) — chosen on the **validation** slice, never on the test set, then read once on test.

In [ ]:
proba = tuned.predict_proba(X_test)[:, 1]
proba_val = tuned.predict_proba(X_val)[:, 1]

# Choose the operating threshold on the VALIDATION slice (never the test set), then
# read it once on test -- the same touch-the-test-once discipline we gave the model.
pv, rv, tv = precision_recall_curve(y_val, proba_val)
f1v = 2 * pv * rv / (pv + rv + 1e-12)
thr_star = float(tv[int(np.argmax(f1v[:-1]))])

prec, rec, _ = precision_recall_curve(y_test, proba)   # the test curve, for the picture
p05 = precision_score(y_test, (proba >= 0.5).astype(int))
r05 = recall_score(y_test, (proba >= 0.5).astype(int))
ps = precision_score(y_test, (proba >= thr_star).astype(int))
rs = recall_score(y_test, (proba >= thr_star).astype(int))

fig, (axa, axb) = plt.subplots(1, 2, figsize=(12, 4.8))
axa.plot(rec, prec, color=COLORS["model"], linewidth=2)
axa.plot(r05, p05, "o", color=COLORS["test"], markersize=12,
         label=f"threshold 0.5  (P {p05:.2f} / R {r05:.2f})")
axa.plot(rs, ps, "o", color=COLORS["highlight"], markersize=12,
         label=f"val-chosen {thr_star:.3f}  (P {ps:.2f} / R {rs:.2f})")
axa.axhline(y_test.mean(), color=COLORS["muted"], linestyle="--",
            label=f"base rate {y_test.mean():.2f}")
axa.set_xlabel("recall")
axa.set_ylabel("precision")
axa.set_title("(a) precision-recall curve (tuned XGB, test)")
axa.legend(loc="lower left")

cm = confusion_matrix(y_test, (proba >= thr_star).astype(int))
axb.imshow(cm, cmap=CMAP_COUNT)
for (r, c), v in np.ndenumerate(cm):
    axb.text(c, r, str(v), ha="center", va="center", color=COLORS["text"])
axb.set_xticks([0, 1])
axb.set_xticklabels(["<=50K", ">50K"])
axb.set_yticks([0, 1])
axb.set_yticklabels(["<=50K", ">50K"])
axb.set_xlabel("predicted")
axb.set_ylabel("true")
axb.set_title(f"(b) confusion @ threshold {thr_star:.3f}")
plt.show()
print(f"threshold 0.5:         precision {p05:.4f}   recall {r05:.4f}")
print(f"val-chosen {thr_star:.3f}:  precision {ps:.4f}   recall {rs:.4f}")


**Read the figure.** At the default **0.5**, the model is fairly **precise** (≈ 0.78 of its `>50K`
predictions are right) but **misses** many high earners (recall ≈ 0.65). Lowering the threshold to the
one we chose on the validation slice (**≈ 0.30**) flips the balance — recall rises to ≈ 0.83 while
precision eases to ≈ 0.65 — so the confusion matrix catches more true `>50K` cases at the cost of more
false alarms. Two honest notes: the threshold is a **choice**, set by the relative cost of a miss versus
a false alarm (there is no single "accuracy" that captures it); and we picked it on **validation, not
the test set** — the same touch-the-test-once discipline we gave the model (Chapter 00).

## What drives the model? Gain importance vs permutation

XGBoost's `feature_importances_` is **gain-based MDI**, with the same bias we met in Chapters 06 and 08.
The trustworthy, held-out cross-check is **permutation importance** — shuffle one feature in the test set
and watch how much PR-AUC drops. They can tell very different stories.

In [ ]:
gain = pd.Series(tuned.feature_importances_, index=X.columns).sort_values(ascending=False).head(6)
perm = permutation_importance(tuned, X_test, y_test, scoring="average_precision",
                              n_repeats=5, random_state=SEED, n_jobs=-1)
perm_s = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False).head(6)

fig, (axa, axb) = plt.subplots(1, 2, figsize=(12, 4.8))
axa.barh(gain.index[::-1], gain.values[::-1], color=COLORS["class_a"])
axa.set_xlabel("normalized gain")
axa.set_title("(a) gain importance (MDI)")
axb.barh(perm_s.index[::-1], perm_s.values[::-1], color=COLORS["model"])
axb.set_xlabel("PR-AUC drop when shuffled")
axb.set_title("(b) permutation importance (held-out)")
plt.show()
print("gain-MDI top:   " + ", ".join(f"{k} {v:.3f}" for k, v in gain.items()))
print("permutation top: " + ", ".join(f"{k} {v:.3f}" for k, v in perm_s.items()))


**Read the figure.** They **disagree sharply**. Gain says `relationship` dominates (≈ 0.34);
permutation says `capital-gain` (≈ 0.24), with `relationship` far down the list (≈ 0.04). The reason is
**redundancy**: `relationship` and `marital-status` carry nearly the same information, so the trees split
on `relationship` often (high gain) — but shuffling it barely hurts, because the signal survives in
`marital-status`. `capital-gain` has no such backup, so permuting it collapses PR-AUC. Trust the
held-out **permutation** read, and remember importance is *use*, not cause — and reflects this dataset,
not the world.

## Limits and ethics — read this before you would ever deploy

We have a competitive model. Now the part that matters most. This model predicts a 1994 income label,
and that label is **itself demographically skewed**. Let us look at the base rates directly.

In [ ]:
by_sex = y.groupby(X["sex"], observed=True).mean()
by_race = y.groupby(X["race"], observed=True).mean().sort_values()
fig, (axa, axb) = plt.subplots(1, 2, figsize=(12, 4.6))
axa.bar(by_sex.index.astype(str), by_sex.values, color=COLORS["class_b"])
axa.axhline(y.mean(), color=COLORS["muted"], linestyle="--", label=f"overall {y.mean():.3f}")
axa.set_ylabel("P(income > 50K)")
axa.set_title("(a) base rate by sex")
axa.legend()
axb.barh(by_race.index.astype(str), by_race.values, color=COLORS["class_b"])
axb.axvline(y.mean(), color=COLORS["muted"], linestyle="--", label=f"overall {y.mean():.3f}")
axb.set_xlabel("P(income > 50K)")
axb.set_title("(b) base rate by race")
axb.legend()
plt.show()
print("P(>50K) by sex:  " + ", ".join(f"{k} {v:.3f}" for k, v in by_sex.items()))
print("P(>50K) by race: " + ", ".join(f"{k} {v:.3f}" for k, v in by_race.items()))


**Read the figure, and the warning in it.** The label encodes the inequities of its time and
place: in this 1994 US data, women are recorded at `>50K` about **11%** of the time versus **30%** for
men, and the rate for Black respondents (≈ 12%) is half that of White respondents (≈ 25%). A model
trained to predict this label **learns and reproduces those disparities** — and features like
`relationship`, `marital-status`, and `sex` act as **demographic proxies**, so even "dropping the
sensitive column" would not remove the effect.

The consequence is concrete: using such a model to gate real decisions — credit, hiring, who sees an
opportunity — would **automate and entrench** historical bias, lending it a false air of objectivity.
This is a teaching dataset for learning the mechanics of tabular modelling; it is **not** something to
deploy. Honest scope: the data is 1994 US, the target is a coarse binary, and "fairness" here has no
single technical fix — it is a question about whether the task should be automated at all. A strong
PR-AUC does not answer that question.

## Your turn

1. **(easy)** Pick a decision threshold for two different goals: a **recall-first** screen (catch as
   many `>50K` as possible) and a **precision-first** alert (be right when you flag). Recompute
   `precision_recall_curve` on the tuned model's test scores, and report the precision and recall each
   threshold gives.
2. **(core)** Add three explicit `is_missing` indicator columns (one per gappy feature) to the native
   model. Predict, before you run it, whether PR-AUC will move — then check. Explain the result using
   what Figure 3 already showed.
3. **(reach)** The XGBoost-vs-GradientBoosting gap (≈ 0.828 vs ≈ 0.814) mixes **two** effects: the model
   and the preprocessing (native categoricals vs ordinal-encoded). Design a comparison that **isolates
   the model** — hint: give both the same ordinal-encoded, imputed inputs — and say what you expect the
   gap to do.

## What you built — and the chapter behind it

You ran a full, honest workflow on real, messy data: you read the imbalance and chose **PR-AUC**; you
found informative missingness and then **measured** that native handling did not beat imputation here
(redundant signal — *measure the lever, do not assume*); you compared the field on disclosed terms
(boosters lead, by a modest, partly-convenience margin — **no universal best**); you tuned with **early
stopping**; you chose a **threshold** with eyes open; you read importances with **permutation**, not
gain alone; and you confronted the model's **ethical limits** plainly.

And that closes Chapter 09. **XGBoost is not a new algorithm** — it is Chapter 08's gradient-boosting
engine, refined and engineered:

- the **second-order view** — every leaf is `w* = −G/(H+λ)` (NB 1);
- the **regularized objective** — `λ` and `γ` priced inside the loss, the gain that decides each split (NB 2);
- **sparsity-aware** splits — a learned default direction for missing values (NB 3);
- the **histogram** engine — binning that makes it fast at no accuracy cost, and the parameters you tune (NB 4);
- and this capstone, where it all meets a real problem (NB 5).

Its honest edges are **speed, native handling of missing values and categoricals, and the regularization
you control** — not a guaranteed jump in accuracy.

**Vocabulary recap.** PR-AUC / average precision · the decision threshold · precision/recall trade-off ·
native missing handling vs imputation · permutation vs gain importance · demographic proxy · early
stopping (`eval_set`, `best_iteration`).

Next: **Chapter 10 — LightGBM**, the leaf-wise sibling you saw match XGBoost in about two seconds.

## References

- Chen, T., & Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System.* KDD '16. DOI
  [10.1145/2939672.2939785](https://doi.org/10.1145/2939672.2939785).
- Kohavi, R. (1996). *Scaling up the accuracy of naive-Bayes classifiers: a decision-tree hybrid.* KDD
  '96. (The Adult / Census Income dataset; drawn from the 1994 US Census.)
- Friedman, J. H. (2001). *Greedy function approximation: a gradient boosting machine.* Annals of
  Statistics, 29(5). DOI [10.1214/aos/1013203451](https://doi.org/10.1214/aos/1013203451).
- Ke, G., Meng, Q., Finley, T., et al. (2017). *LightGBM: A Highly Efficient Gradient Boosting Decision
  Tree.* NeurIPS 2017. (Chapter 10.)
- Pedregosa, F., et al. (2011). *Scikit-learn: Machine Learning in Python.* JMLR 12, 2825–2830.
  (Average precision / PR-AUC, permutation importance, `ColumnTransformer`.)
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.),
  §10. DOI [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7).

*Previous: 04 — The estimator and its parameters.*  ·  *Chapter 09 complete.*